# Day 2 — VisDrone smoke training dan bukti resume

Notebook ini hanya menjalankan smoke training 3 epoch pada subset deterministik (256 train, 64 val). Phase 1 berhenti setelah epoch 2, menyimpan checkpoint mentah berisi optimizer, lalu phase 2 benar-benar resume pada epoch 3. Ini bukan full fine-tuning.

Prasyarat: runtime Colab GPU, secret Colab `GITHUB_TOKEN` dengan akses ke repository private, serta dua ZIP resmi di `MyDrive/multi-uav-perception/data/raw/visdrone2019_det/`. Jangan menempelkan token ke cell.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os, stat, subprocess, sys

drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Tambahkan GITHUB_TOKEN ke Colab Secrets lalu jalankan ulang.')

repo = Path('/content/multi-uav-perception')
if repo.exists():
    raise RuntimeError('Direktori repo sudah ada. Gunakan runtime baru agar run tetap immutable.')
askpass = Path('/content/codex_git_askpass.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) echo "x-access-token" ;;;;;\n*) echo "$GITHUB_TOKEN" ;;;;;\nesac\n'.replace(';;;;;', ';;'), encoding='utf-8')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
git_env = os.environ.copy()
git_env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': token})
subprocess.run(['git', 'clone', '--branch', 'main', '--single-branch', 'https://github.com/muqsithanif/multi-uav-perception.git', str(repo)], check=True, env=git_env)
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip()
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
print({'repo': str(repo), 'revision': revision})

In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'ultralytics==8.4.115', 'PyYAML==6.0.3', 'gdown==6.1.0'], check=True)
import torch, ultralytics, yaml
if not torch.cuda.is_available():
    raise RuntimeError('GPU CUDA tidak tersedia. Pilih Runtime > Change runtime type > GPU.')
print({'torch': torch.__version__, 'ultralytics': ultralytics.__version__, 'gpu': torch.cuda.get_device_name(0)})

In [ ]:
import hashlib, shutil
from scripts.download_visdrone import extract_archive, validate_extracted_split

drive_raw = Path('/content/drive/MyDrive/multi-uav-perception/data/raw/visdrone2019_det')
repo_raw = repo / 'data/raw/visdrone2019_det'
repo_raw.mkdir(parents=True, exist_ok=True)
archives = {
    'train': ('VisDrone2019-DET-train.zip', 'VisDrone2019-DET-train', 6471, '86a77eba93137bfc16e4993860de9245b0675c0dba0d3ab98fb458699e256f84'),
    'val': ('VisDrone2019-DET-val.zip', 'VisDrone2019-DET-val', 548, 'abeea063037e5d20398837deb11084e652402a34ddf4f207bdf541a6f2a35ef9'),
}
for split, (filename, dirname, expected_images, expected_sha) in archives.items():
    source = drive_raw / filename
    if not source.is_file():
        raise FileNotFoundError(f'ZIP belum ada di Google Drive: {source}')
    destination = repo_raw / filename
    shutil.copy2(source, destination)
    digest = hashlib.sha256()
    with destination.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    if digest.hexdigest() != expected_sha:
        raise ValueError(f'SHA-256 {split} tidak cocok: {digest.hexdigest()}')
    extracted = repo_raw / dirname
    extract_archive(destination, repo_raw, extracted)
    validate_extracted_split(extracted, expected_images)

base_config = yaml.safe_load((repo / 'configs/visdrone_conversion.yaml').read_text(encoding='utf-8'))
base_config['manifest_path'] = 'data/processed/colab_conversion_manifest.json'
base_config['validation_report_path'] = 'data/processed/colab_conversion_report.json'
colab_config = repo / 'data/processed/visdrone_conversion_colab.yaml'
colab_config.parent.mkdir(parents=True, exist_ok=True)
colab_config.write_text(yaml.safe_dump(base_config, sort_keys=False), encoding='utf-8')
subprocess.run([sys.executable, 'scripts/prepare_visdrone.py', '--config', str(colab_config)], cwd=repo, check=True)
print('Dataset resmi berhasil diverifikasi dan dikonversi.')

In [ ]:
test_files = [
    'tests/test_visdrone_smoke_training.py',
    'tests/test_pretrained_baseline.py',
    'tests/test_visdrone_dataset.py',
    'tests/test_visdrone_audit.py',
    'tests/test_visdrone_distribution.py',
]
subprocess.run([sys.executable, '-m', 'pytest', '-q', *test_files], cwd=repo, check=True)

In [ ]:
persistent_root = Path('/content/drive/MyDrive/multi-uav-perception/training')
persistent_root.mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable, 'scripts/run_visdrone_smoke_training.py',
    '--config', 'configs/visdrone_smoke_train.yaml',
    '--persistent-root', str(persistent_root),
], cwd=repo, check=True)

In [ ]:
import json
experiment_id = 'S01_20260807_colab_smoke'
summary_path = repo / 'experiments' / experiment_id / 'summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
if summary['status'] != 'passed' or summary['resume_proof']['status'] != 'passed':
    raise RuntimeError('Smoke training atau resume proof belum lulus.')
print(json.dumps({
    'status': summary['status'],
    'resume_proof': summary['resume_proof'],
    'persistent_artifacts': summary['persistent_artifacts'],
    'smoke_metrics_last_epoch': summary['smoke_metrics_last_epoch'],
}, indent=2))

allowed = (f'experiments/{experiment_id}/', f'results/day2/{experiment_id}/')
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=repo, text=True).splitlines()
unexpected = [line for line in dirty if not line[3:].replace('\\', '/').startswith(allowed)]
if unexpected:
    raise RuntimeError(f'Perubahan tak terduga sebelum push: {unexpected}')
branch = f'colab/day2-{experiment_id}'
subprocess.run(['git', 'config', 'user.name', 'muqsithanif'], cwd=repo, check=True)
subprocess.run(['git', 'config', 'user.email', 'muqsithanif29@gmail.com'], cwd=repo, check=True)
subprocess.run(['git', 'switch', '-c', branch], cwd=repo, check=True)
subprocess.run(['git', 'add', '--', allowed[0], allowed[1]], cwd=repo, check=True)
subprocess.run(['git', 'commit', '-m', 'train: record Colab VisDrone smoke resume'], cwd=repo, check=True)
subprocess.run(['git', 'push', '-u', 'origin', branch], cwd=repo, env=git_env, check=True)
print({'pushed_branch': branch, 'gate_2a_candidate': True})